# Minimal Model phase diagram in $(K, l)$

Reproduces `notebooks/forfigures/MM 1 - phase diagrams.ipynb` from the Julia codebase.

For each grid cell $(K, l)$ we (i) compute the closed-form non-spatial steady states of the Minimal Model, (ii) pick the larger-$N$ branch, (iii) run a wavenumber scan, and (iv) classify the result as **extinct**, **stable**, or **Turing-unstable**. The analytical extinction line `fr_ext_line_K` and the instability lines `fr_cor1_instab_line_K` for several diffusion ratios $p = D_R/D_I$ are overlaid on top.

In [ ]:
import sys, pathlib
_root = pathlib.Path.cwd().parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_context("talk", rc={"font.size": 15, "axes.titlesize": 15, "axes.labelsize": 15})
sns.set_style("whitegrid", {"grid.color": '.9', 'grid.linestyle': '--', 'axes.edgecolor': '.6', 'xtick.bottom': True, 'ytick.left': True})

from ssmc.minimal_model import MMParams, mm_get_nospace_sol, mmp_to_smicrm
from ssmc.linstab import scan_k, classify
from ssmc.plotting import setup_mm_Kl_ax, draw_fr_ext_line, draw_fr_cor1_instab_line, STATE_COLORS

In [ ]:
def classify_cell(K, l, p, m=1.0, c=1.0, ks=None):
    """Return one of 'extinct', 'stable', 'unstable' for a single (K, l, p) cell."""
    if ks is None:
        ks = np.linspace(1e-3, 30.0, 200)
    mmp = MMParams(K=K, m=m, c=c, l=l)
    sols = [s for s in mm_get_nospace_sol(mmp) if s[0] > 1e-8]
    if not sols:
        return 'extinct'
    sols.sort(key=lambda s: s[0], reverse=True)
    sparams = mmp_to_smicrm(mmp, DN=1e-12, DI=1.0, DR=p)
    res = scan_k(sparams, sols[0], ks)
    return classify(res)

def phase_grid(Ks, ls, p, **kwargs):
    states = np.empty((len(ls), len(Ks)), dtype=object)
    for j, K in enumerate(Ks):
        for i, l in enumerate(ls):
            states[i, j] = classify_cell(float(K), float(l), p, **kwargs)
    return states

STATE_TO_INT = {'extinct': 0, 'stable': 1, 'unstable': 2, 'nospace_unstable': 3}
INT_TO_STATE = {v: k for k, v in STATE_TO_INT.items()}

In [ ]:
Ks = np.geomspace(0.1, 100.0, 50)
ls = np.linspace(0.02, 0.98, 50)
ps = [1.0, 0.1, 0.01]

grids = {p: phase_grid(Ks, ls, p) for p in ps}

In [ ]:
from matplotlib.colors import ListedColormap

cmap = ListedColormap([STATE_COLORS['extinct'], STATE_COLORS['stable'], STATE_COLORS['unstable'], STATE_COLORS['nospace_unstable']])

fig, axes = plt.subplots(1, len(ps), figsize=(5 * len(ps), 4.5), constrained_layout=True)
for ax, p in zip(axes, ps):
    grid = np.vectorize(STATE_TO_INT.get)(grids[p]).astype(float)
    ax.pcolormesh(Ks, ls, grid, cmap=cmap, vmin=-0.5, vmax=3.5, shading='auto')
    setup_mm_Kl_ax(ax, K_lim=(Ks[0], Ks[-1]))
    draw_fr_ext_line(ax, color='k', linestyle='-')
    draw_fr_cor1_instab_line(ax, p=p, color='k', linestyle='--')
    ax.set_title(f"$p = D_R/D_I = {p}$")

from matplotlib.patches import Patch
handles = [Patch(facecolor=STATE_COLORS[s], label=s) for s in ('extinct', 'stable', 'unstable')]
fig.legend(handles=handles, loc='lower center', ncol=3, bbox_to_anchor=(0.5, -0.05))
fig.suptitle("Minimal Model phase diagram", y=1.02)
plt.show()

**Sanity check.** The black solid curve (extinction) should hug the boundary between blue and orange. The black dashed curve (instability for the corresponding $p$) should hug the boundary between orange and green and shift left as $p$ shrinks (more disparate diffusion → easier to destabilise).